# Pandas 시작하기: Series와 DataFrame 실습

> 실습 주제: **Pandas 시작하기, Series, DataFrame**  
> 활용 데이터: `boston.csv`, `cancer.csv`, `iris.data`, `wdbc.data`, `ram_price.csv`, `shampoo.csv`, `stock_px.csv`

단순 예제 데이터 대신, 이미 제공된 실제 데이터 파일을 활용하여 **읽기 → 구조 확인 → 선택/가공 → 결측 처리 → 요약 통계 → 상관관계 확인**까지 이어지는 기본 분석 흐름을 연습합니다.

## 0. 실습 목표와 강의 내용 연계

| 강의 내용 | 실습에서 다루는 기능 | 사용 데이터 |
|---|---|---|
| Pandas와 PyData 흐름 | 데이터 로딩, 분석 흐름 이해 | 전체 데이터 |
| Series | `pd.Series`, `index`, `values`, `dtype`, `name`, `index.name` | Boston 가격, 주가 데이터 |
| DataFrame | 생성 방식, `index`, `columns`, `values`, `dtypes`, `shape`, `ndim`, `size` | Boston, Iris, Cancer |
| DataFrame 생성 | 딕셔너리, 리스트, NumPy 배열, Series 사전 | 실제 데이터 일부 |
| 색인과 컬럼 | 행 인덱스, 열 인덱스, 이름 지정 | Boston, Iris |
| 컬럼 선택/추가/삭제 | 단일 열, 복수 열, 파생변수, `del`, `drop` | Boston |
| `loc`, `iloc` | 라벨 기반/위치 기반 인덱싱 | Iris |
| Boolean indexing | 조건 필터링, `isin` | Iris, Cancer |
| NaN 처리 | `isnull`, `notnull`, `dropna`, `fillna`, 결측 flag | Cancer |
| 산술 연산과 정렬 | 인덱스 정렬, `add(fill_value=...)` | Stock |
| `apply` | 열/행 방향 함수 적용, 사용자 정의 함수 | Boston |
| 정렬과 순위 | `sort_index`, `sort_values`, `rank` | Boston, Stock |
| 기술통계 | `describe`, `count`, `mean`, `median`, `var`, `std`, `diff`, `quantile` | Boston, Shampoo, Stock, RAM |
| 유일값 처리 | `unique`, `value_counts`, `isin` | Iris, Cancer, WDBC |
| 상관관계와 공분산 | `corr`, `corrwith`, `cov` | Boston, Stock, Cancer |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = Path("./data")

required_files = [
    "boston.csv",
    "cancer.csv",
    "iris.data",
    "wdbc.data",
    "ram_price.csv",
    "shampoo.csv",
    "stock_px.csv",
]

file_info = pd.DataFrame({
    "file": required_files,
    "size_kb": [round((DATA_DIR / name).stat().st_size / 1024, 2) for name in required_files]
})
file_info

,file,size_kb
0,boston.csv,38.7500
1,cancer.csv,122.4900
2,iris.data,4.4400
3,wdbc.data,147.3100
4,ram_price.csv,5.1100
5,shampoo.csv,0.5900
6,stock_px.csv,99.0300


## 2. 실제 데이터 로딩

강의에서는 `pd.read_csv()`와 같은 입력 함수를 통해 파일을 DataFrame으로 읽어오는 흐름을 강조합니다.  
여기서는 파일마다 형식이 조금씩 다르므로, 열 이름이 없는 파일에는 직접 컬럼명을 부여합니다.

In [2]:
# 1) 기본 CSV: 이미 컬럼명이 있는 데이터
boston = pd.read_csv(DATA_DIR / "boston.csv")
cancer = pd.read_csv(DATA_DIR / "cancer.csv")
ram_price = pd.read_csv(DATA_DIR / "ram_price.csv")

# 2) 컬럼명이 없는 Iris 데이터: 직접 컬럼명 지정
iris_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
iris = pd.read_csv(DATA_DIR / "iris.data", header=None, names=iris_cols)

# 3) WDBC 원본 데이터: id, diagnosis와 30개 특성명 지정
wdbc_features = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
    "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
    "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst",
]
wdbc = pd.read_csv(DATA_DIR / "wdbc.data", header=None, names=["id", "diagnosis"] + wdbc_features)

# 4) 시계열 형태 데이터
stock = pd.read_csv(DATA_DIR / "stock_px.csv", index_col=0, parse_dates=True)
stock.index.name = "date"

shampoo = pd.read_csv(DATA_DIR / "shampoo.csv")
shampoo = shampoo.rename(columns={"Sales of shampoo over a three year period": "sales"})

# 일부 배포본에는 파일 끝에 설명 문장/빈 줄이 들어 있을 수 있으므로 실제 월 데이터만 남깁니다.
shampoo = shampoo[shampoo["Month"].astype(str).str.contains("-", na=False)].copy()
shampoo["sales"] = pd.to_numeric(shampoo["sales"], errors="coerce")

def parse_shampoo_month(value):
    year_code, month = str(value).split("-")
    return pd.Timestamp(year=2000 + int(year_code), month=int(month), day=1)

shampoo["date"] = shampoo["Month"].map(parse_shampoo_month)
shampoo = shampoo.set_index("date")

# 데이터셋 크기 요약
datasets = {
    "boston": boston,
    "cancer": cancer,
    "iris": iris,
    "wdbc": wdbc,
    "ram_price": ram_price,
    "shampoo": shampoo,
    "stock": stock,
}

shape_summary = pd.DataFrame(
    {name: {"rows": df.shape[0], "columns": df.shape[1]} for name, df in datasets.items()}
).T
shape_summary

,rows,columns
boston,506,14
cancer,569,31
iris,150,5
wdbc,569,32
ram_price,333,2
shampoo,36,2
stock,2214,4


In [3]:
# 각 데이터의 첫 3행 확인
for name, df in datasets.items():
    print(f"\n[{name}] shape={df.shape}")
    display(df.head(3))


[boston] shape=(506, 14)


,Price,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,24.0000,0.0063,18.0000,2.3100,0.0000,0.5380,6.5750,65.2000,4.0900,1.0000,296.0000,15.3000,396.9000,4.9800
1,21.6000,0.0273,0.0000,7.0700,0.0000,0.4690,6.4210,78.9000,4.9671,2.0000,242.0000,17.8000,396.9000,9.1400
2,34.7000,0.0273,0.0000,7.0700,0.0000,0.4690,7.1850,61.1000,4.9671,2.0000,242.0000,17.8000,392.8300,4.0300



[cancer] shape=(569, 31)


,type,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,malignant,17.9900,10.3800,122.8000,1001.0000,0.1184,0.2776,0.3001,0.1471,0.2419,0.0787,1.0950,0.9053,8.5890,153.4000,0.0064,0.0490,0.0537,0.0159,0.0300,0.0062,25.3800,17.3300,184.6000,2019.0000,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189
1,malignant,20.5700,17.7700,132.9000,1326.0000,0.0847,0.0786,0.0869,0.0702,0.1812,0.0567,0.5435,0.7339,3.3980,74.0800,0.0052,0.0131,0.0186,0.0134,0.0139,0.0035,24.9900,23.4100,158.8000,1956.0000,0.1238,0.1866,0.2416,0.1860,0.2750,0.0890
2,malignant,19.6900,21.2500,130.0000,1203.0000,0.1096,0.1599,0.1974,0.1279,0.2069,0.0600,0.7456,0.7869,4.5850,94.0300,0.0062,0.0401,0.0383,0.0206,0.0225,0.0046,23.5700,25.5300,152.5000,1709.0000,0.1444,0.4245,0.4504,0.2430,0.3613,0.0876



[iris] shape=(150, 5)


...

,date,price
0,1957.0000,411041792.0000
1,1959.0000,67947725.0000
2,1960.0000,5242880.0000



[shampoo] shape=(36, 2)


,Month,sales
date,,
2001-01-01,1-01,266.0000
2001-02-01,1-02,145.9000
2001-03-01,1-03,183.1000



[stock] shape=(2214, 4)


,AAPL,MSFT,XOM,SPX
date,,,,
2003-01-02,7.4000,21.1100,29.2200,909.0300
2003-01-03,7.4500,21.1400,29.2400,908.5900
2003-01-06,7.4500,21.5200,29.9600,929.0100


## 3. Series 이해하기

Series는 **1차원 배열 + 인덱스**로 이해할 수 있습니다.  
강의 내용처럼 Series는 `index`, `values`, `dtype`, `name`, `index.name`과 같은 속성을 가집니다.

In [4]:
# Boston 데이터의 주택 가격 컬럼을 Series로 추출
price_series = boston["Price"].copy()
price_series.name = "Boston_Price"
price_series.index.name = "record_id"

print("객체 타입:", type(price_series))
display(price_series.head())

series_info = pd.Series({
    "index_type": type(price_series.index).__name__,
    "values_type": type(price_series.values).__name__,
    "dtype": str(price_series.dtype),
    "name": price_series.name,
    "index_name": price_series.index.name,
    "length": len(price_series),
})
series_info

객체 타입: <class 'pandas.core.series.Series'>


record_id
0   24.0000
1   21.6000
2   34.7000
3   33.4000
4   36.2000
Name: Boston_Price, dtype: float64

index_type       RangeIndex
values_type         ndarray
dtype               float64
name           Boston_Price
index_name        record_id
length                  506
dtype: object

In [5]:
# 시계열 Series: 날짜 인덱스를 가진 AAPL 주가
apple = stock["AAPL"].copy()
apple.name = "AAPL_price"

print("AAPL Series의 인덱스 타입:", type(apple.index).__name__)
display(apple.head())
display(apple.tail())

AAPL Series의 인덱스 타입: DatetimeIndex


date
2003-01-02   7.4000
2003-01-03   7.4500
2003-01-06   7.4500
2003-01-07   7.4300
2003-01-08   7.2800
Name: AAPL_price, dtype: float64

date
2011-10-10   388.8100
2011-10-11   400.2900
2011-10-12   402.1900
2011-10-13   408.4300
2011-10-14   422.0000
Name: AAPL_price, dtype: float64

In [6]:
# Series 직접 생성: Iris 품종별 빈도에서 Series 생성
species_counts = iris["species"].value_counts()
species_counts.name = "count"
species_counts.index.name = "species"

species_counts

species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

### 실습 3-1. Series 확인 문제

아래 셀을 실행한 뒤, 다음 질문에 답해보세요.

1. `ram_price["price"]`는 Series인가요, DataFrame인가요?
2. `ram_price[["price"]]`는 Series인가요, DataFrame인가요?
3. 두 객체의 `shape`는 어떻게 다른가요?

In [7]:
ram_price_series = ram_price["price"]
ram_price_frame = ram_price[["price"]]

print(type(ram_price_series), ram_price_series.shape)
print(type(ram_price_frame), ram_price_frame.shape)

display(ram_price_series.head())
display(ram_price_frame.head())

<class 'pandas.core.series.Series'> (333,)
<class 'pandas.core.frame.DataFrame'> (333, 1)


0   411041792.0000
1    67947725.0000
2     5242880.0000
3     2642412.0000
4      734003.0000
Name: price, dtype: float64

,price
0,411041792.0000
1,67947725.0000
2,5242880.0000
3,2642412.0000
4,734003.0000


## 4. DataFrame 이해하기

DataFrame은 표와 같은 **2차원 자료구조**입니다.  
행 방향 인덱스는 `index`, 열 방향 인덱스는 `columns`로 확인합니다.

In [8]:
# Boston 데이터 일부를 이용하여 DataFrame의 주요 속성 확인
boston_sample = boston.head(5).copy()
boston_sample.index.name = "record_id"
boston_sample.columns.name = "variable"

display(boston_sample)

attributes = pd.Series({
    "index": str(boston_sample.index),
    "columns_sample": list(boston_sample.columns[:5]),
    "values_type": type(boston_sample.values).__name__,
    "shape": boston_sample.shape,
    "ndim": boston_sample.ndim,
    "size": boston_sample.size,
})
attributes

variable,Price,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
record_id,,,,,,,,,,,,,,
0,24.0000,0.0063,18.0000,2.3100,0.0000,0.5380,6.5750,65.2000,4.0900,1.0000,296.0000,15.3000,396.9000,4.9800
1,21.6000,0.0273,0.0000,7.0700,0.0000,0.4690,6.4210,78.9000,4.9671,2.0000,242.0000,17.8000,396.9000,9.1400
2,34.7000,0.0273,0.0000,7.0700,0.0000,0.4690,7.1850,61.1000,4.9671,2.0000,242.0000,17.8000,392.8300,4.0300
3,33.4000,0.0324,0.0000,2.1800,0.0000,0.4580,6.9980,45.8000,6.0622,3.0000,222.0000,18.7000,394.6300,2.9400
4,36.2000,0.0691,0.0000,2.1800,0.0000,0.4580,7.1470,54.2000,6.0622,3.0000,222.0000,18.7000,396.9000,5.3300


index             RangeIndex(start=0, stop=5, step=1, name='reco...
columns_sample                       [Price, CRIM, ZN, INDUS, CHAS]
values_type                                                 ndarray
shape                                                       (5, 14)
ndim                                                              2
size                                                             70
dtype: object

In [9]:
# DataFrame의 dtypes 확인
boston_sample.dtypes

variable
Price      float64
CRIM       float64
ZN         float64
INDUS      float64
CHAS       float64
NOX        float64
RM         float64
AGE        float64
DIS        float64
RAD        float64
TAX        float64
PTRATIO    float64
B          float64
LSTAT      float64
dtype: object

In [10]:
# DataFrame 전체 정보 요약
boston.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    506 non-null    float64
 1   CRIM     506 non-null    float64
 2   ZN       506 non-null    float64
 3   INDUS    506 non-null    float64
 4   CHAS     506 non-null    float64
 5   NOX      506 non-null    float64
 6   RM       506 non-null    float64
 7   AGE      506 non-null    float64
 8   DIS      506 non-null    float64
 9   RAD      506 non-null    float64
 10  TAX      506 non-null    float64
 11  PTRATIO  506 non-null    float64
 12  B        506 non-null    float64
 13  LSTAT    506 non-null    float64
dtypes: float64(14)
memory usage: 55.5 KB


## 5. DataFrame 생성 방법

강의에서 언급한 여러 DataFrame 생성 방법을 실제 데이터 일부로 재구성합니다.

In [11]:
# 1) 딕셔너리로 DataFrame 생성
summary_from_dict = pd.DataFrame({
    "dataset": ["boston", "cancer", "iris", "stock"],
    "rows": [len(boston), len(cancer), len(iris), len(stock)],
    "columns": [boston.shape[1], cancer.shape[1], iris.shape[1], stock.shape[1]],
    "main_target_or_key": ["Price", "type", "species", "date index"],
})
summary_from_dict

,dataset,rows,columns,main_target_or_key
0,boston,506,14,Price
1,cancer,569,31,type
2,iris,150,5,species
3,stock,2214,4,date index


In [12]:
# 2) 딕셔너리의 리스트로 DataFrame 생성
records = iris.head(3).to_dict("records")
records_from_list = pd.DataFrame(records)
records_from_list

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1000,3.5000,1.4000,0.2000,Iris-setosa
1,4.9000,3.0000,1.4000,0.2000,Iris-setosa
2,4.7000,3.2000,1.3000,0.2000,Iris-setosa


In [13]:
# 3) NumPy 2차원 ndarray로 DataFrame 생성
boston_array = boston[["Price", "RM", "LSTAT"]].head(5).to_numpy()
from_ndarray = pd.DataFrame(boston_array, columns=["Price", "RM", "LSTAT"])
from_ndarray

,Price,RM,LSTAT
0,24.0000,6.5750,4.9800
1,21.6000,6.4210,9.1400
2,34.7000,7.1850,4.0300
3,33.4000,6.9980,2.9400
4,36.2000,7.1470,5.3300


In [14]:
# 4) Series 사전으로 DataFrame 생성
stock_series_frame = pd.DataFrame({
    "AAPL": stock["AAPL"].head(5),
    "MSFT": stock["MSFT"].head(5),
    "XOM": stock["XOM"].head(5),
})
stock_series_frame

,AAPL,MSFT,XOM
date,,,
2003-01-02,7.4000,21.1100,29.2200
2003-01-03,7.4500,21.1400,29.2400
2003-01-06,7.4500,21.5200,29.9600
2003-01-07,7.4300,21.9300,28.9500
2003-01-08,7.2800,21.3100,28.8300


## 6. 컬럼 선택, 추가, 삭제

DataFrame에서는 `df['컬럼명']`으로 단일 열을 선택하면 Series가 반환되고, `df[['컬럼1', '컬럼2']]`처럼 리스트로 선택하면 DataFrame이 반환됩니다.

In [15]:
# 단일 컬럼 선택: Series 반환
single_col = boston["Price"]
print(type(single_col))
display(single_col.head())

# 복수 컬럼 선택: DataFrame 반환
multi_col = boston[["Price", "RM", "LSTAT"]]
print(type(multi_col))
display(multi_col.head())

<class 'pandas.core.series.Series'>


0   24.0000
1   21.6000
2   34.7000
3   33.4000
4   36.2000
Name: Price, dtype: float64

<class 'pandas.core.frame.DataFrame'>


,Price,RM,LSTAT
0,24.0000,6.5750,4.9800
1,21.6000,6.4210,9.1400
2,34.7000,7.1850,4.0300
3,33.4000,6.9980,2.9400
4,36.2000,7.1470,5.3300


In [16]:
# 파생 컬럼 만들기
boston_work = boston[["Price", "RM", "LSTAT", "AGE", "TAX"]].copy()

boston_work["price_per_room"] = boston_work["Price"] / boston_work["RM"]
boston_work["high_price"] = boston_work["Price"] > boston_work["Price"].median()
boston_work["tax_level"] = np.where(boston_work["TAX"] >= boston_work["TAX"].median(), "high_tax", "low_tax")

boston_work.head()

,Price,RM,LSTAT,AGE,TAX,price_per_room,high_price,tax_level
0,24.0000,6.5750,4.9800,65.2000,296.0000,3.6502,True,low_tax
1,21.6000,6.4210,9.1400,78.9000,242.0000,3.3640,True,low_tax
2,34.7000,7.1850,4.0300,61.1000,242.0000,4.8295,True,low_tax
3,33.4000,6.9980,2.9400,45.8000,222.0000,4.7728,True,low_tax
4,36.2000,7.1470,5.3300,54.2000,222.0000,5.0651,True,low_tax


In [17]:
# Series를 이용한 컬럼 입력: 인덱스가 일치하는 행에만 값이 들어감
manual_score = pd.Series([100, 80, 60], index=[0, 10, 500])
boston_work["manual_score"] = manual_score

boston_work.loc[[0, 1, 10, 500], ["Price", "manual_score"]]

,Price,manual_score
0,24.0000,100.0000
1,21.6000,NaN
10,15.0000,80.0000
500,16.8000,60.0000


In [18]:
# 열 삭제 예시: del과 drop
boston_deleted = boston_work.copy()
del boston_deleted["manual_score"]

boston_dropped = boston_work.drop(columns=["manual_score"])

print("del 후 컬럼 수:", boston_deleted.shape[1])
print("drop 후 컬럼 수:", boston_dropped.shape[1])

del 후 컬럼 수: 8
drop 후 컬럼 수: 8


## 7. `loc`와 `iloc`로 행/열 선택하기

- `loc`: 라벨 기반 인덱싱
- `iloc`: 정수 위치 기반 인덱싱

Iris 데이터에 문자열 인덱스를 부여한 뒤 두 방식의 차이를 확인합니다.

In [19]:
iris_indexed = iris.copy()
iris_indexed.index = [f"iris_{i:03d}" for i in range(len(iris_indexed))]
iris_indexed.index.name = "sample_id"
iris_indexed.head()

,sepal_length,sepal_width,petal_length,petal_width,species
sample_id,,,,,
iris_000,5.1000,3.5000,1.4000,0.2000,Iris-setosa
iris_001,4.9000,3.0000,1.4000,0.2000,Iris-setosa
iris_002,4.7000,3.2000,1.3000,0.2000,Iris-setosa
iris_003,4.6000,3.1000,1.5000,0.2000,Iris-setosa
iris_004,5.0000,3.6000,1.4000,0.2000,Iris-setosa


In [20]:
# loc: 라벨 기반 선택
iris_indexed.loc["iris_000":"iris_005", ["sepal_length", "sepal_width", "species"]]

,sepal_length,sepal_width,species
sample_id,,,
iris_000,5.1000,3.5000,Iris-setosa
iris_001,4.9000,3.0000,Iris-setosa
iris_002,4.7000,3.2000,Iris-setosa
iris_003,4.6000,3.1000,Iris-setosa
iris_004,5.0000,3.6000,Iris-setosa
iris_005,5.4000,3.9000,Iris-setosa


In [21]:
# iloc: 정수 위치 기반 선택
iris_indexed.iloc[0:6, [0, 1, 4]]

,sepal_length,sepal_width,species
sample_id,,,
iris_000,5.1000,3.5000,Iris-setosa
iris_001,4.9000,3.0000,Iris-setosa
iris_002,4.7000,3.2000,Iris-setosa
iris_003,4.6000,3.1000,Iris-setosa
iris_004,5.0000,3.6000,Iris-setosa
iris_005,5.4000,3.9000,Iris-setosa


In [22]:
# Boolean indexing: 조건이 True인 행만 선택
wide_setosa = iris_indexed[
    (iris_indexed["species"] == "Iris-setosa") &
    (iris_indexed["sepal_width"] >= 3.8)
]
wide_setosa.head(10)

,sepal_length,sepal_width,petal_length,petal_width,species
sample_id,,,,,
iris_005,5.4000,3.9000,1.7000,0.4000,Iris-setosa
iris_014,5.8000,4.0000,1.2000,0.2000,Iris-setosa
iris_015,5.7000,4.4000,1.5000,0.4000,Iris-setosa
iris_016,5.4000,3.9000,1.3000,0.4000,Iris-setosa
iris_018,5.7000,3.8000,1.7000,0.3000,Iris-setosa
iris_019,5.1000,3.8000,1.5000,0.3000,Iris-setosa
iris_032,5.2000,4.1000,1.5000,0.1000,Iris-setosa
iris_033,5.5000,4.2000,1.4000,0.2000,Iris-setosa
iris_044,5.1000,3.8000,1.9000,0.4000,Iris-setosa


In [23]:
# isin: 여러 범주 중 하나에 속하는 행 선택
iris_two_species = iris_indexed[iris_indexed["species"].isin(["Iris-setosa", "Iris-versicolor"])]
iris_two_species["species"].value_counts()

species
Iris-setosa        50
Iris-versicolor    50
Name: count, dtype: int64

## 8. 결측값 NaN 처리

실제 데이터에 결측이 없는 경우도 있으므로, Cancer 데이터 일부에 의도적으로 결측을 만들고 처리 과정을 실습합니다.

핵심 함수:

- `isnull()` / `notnull()`
- `dropna()`
- `fillna()`
- 결측 여부를 별도 flag 컬럼으로 보존

In [24]:
# 원본 결측 개수 확인
pd.DataFrame({
    "dataset": ["boston", "cancer", "iris", "wdbc", "ram_price", "shampoo", "stock"],
    "missing_cells": [df.isnull().sum().sum() for df in [boston, cancer, iris, wdbc, ram_price, shampoo, stock]],
})

,dataset,missing_cells
0,boston,0
1,cancer,0
2,iris,0
3,wdbc,0
4,ram_price,0
5,shampoo,0
6,stock,0


In [25]:
# 실습용 결측 데이터 생성
cancer_na = cancer.copy()
cancer_na.loc[[0, 5, 10], "mean radius"] = np.nan
cancer_na.loc[[1, 5], "mean texture"] = np.nan
cancer_na.loc[[2], "mean area"] = np.nan

cancer_na[["type", "mean radius", "mean texture", "mean area"]].head(12)

,type,mean radius,mean texture,mean area
0,malignant,NaN,10.3800,1001.0000
1,malignant,20.5700,NaN,1326.0000
2,malignant,19.6900,21.2500,NaN
3,malignant,11.4200,20.3800,386.1000
4,malignant,20.2900,14.3400,1297.0000
5,malignant,NaN,NaN,477.1000
6,malignant,18.2500,19.9800,1040.0000
7,malignant,13.7100,20.8300,577.9000
8,malignant,13.0000,21.8200,519.8000
9,malignant,12.4600,24.0400,475.9000


In [26]:
# 결측 여부 확인
missing_summary = cancer_na[["mean radius", "mean texture", "mean area"]].isnull().sum()
missing_summary

mean radius     3
mean texture    2
mean area       1
dtype: int64

In [27]:
# notnull로 결측이 아닌 행만 확인
valid_radius_rows = cancer_na[cancer_na["mean radius"].notnull()]
print("mean radius가 결측이 아닌 행 수:", len(valid_radius_rows))

mean radius가 결측이 아닌 행 수: 566


In [28]:
# dropna: 결측이 있는 행 제거
before_shape = cancer_na.shape
after_drop = cancer_na.dropna(subset=["mean radius", "mean texture", "mean area"])
after_shape = after_drop.shape

print("dropna 전:", before_shape)
print("dropna 후:", after_shape)

dropna 전: (569, 31)
dropna 후: (564, 31)


In [29]:
# fillna: 중앙값 또는 평균으로 결측값 채우기
cancer_filled = cancer_na.copy()
cancer_filled["mean radius_missing"] = cancer_filled["mean radius"].isnull()

cancer_filled["mean radius"] = cancer_filled["mean radius"].fillna(cancer_filled["mean radius"].median())
cancer_filled["mean texture"] = cancer_filled["mean texture"].fillna(cancer_filled["mean texture"].mean())
cancer_filled["mean area"] = cancer_filled["mean area"].fillna(cancer_filled["mean area"].median())

cancer_filled[["type", "mean radius", "mean radius_missing", "mean texture", "mean area"]].head(12)

,type,mean radius,mean radius_missing,mean texture,mean area
0,malignant,13.3550,True,10.3800,1001.0000
1,malignant,20.5700,False,19.2987,1326.0000
2,malignant,19.6900,False,21.2500,548.7500
3,malignant,11.4200,False,20.3800,386.1000
4,malignant,20.2900,False,14.3400,1297.0000
5,malignant,13.3550,True,19.2987,477.1000
6,malignant,18.2500,False,19.9800,1040.0000
7,malignant,13.7100,False,20.8300,577.9000
8,malignant,13.0000,False,21.8200,519.8000
9,malignant,12.4600,False,24.0400,475.9000


## 9. 산술 연산과 인덱스 정렬

pandas는 Series나 DataFrame끼리 연산할 때 **인덱스를 기준으로 자동 정렬**합니다.  
겹치지 않는 인덱스는 NaN이 됩니다.

In [30]:
# 날짜 인덱스가 서로 완전히 같지 않은 두 Series 만들기
aapl_first = stock["AAPL"].iloc[:5]
msft_shifted = stock["MSFT"].iloc[2:7]

alignment_demo = pd.DataFrame({
    "AAPL_first": aapl_first,
    "MSFT_shifted": msft_shifted,
    "AAPL_plus_MSFT": aapl_first + msft_shifted,
    "AAPL_plus_MSFT_fill0": aapl_first.add(msft_shifted, fill_value=0),
})
alignment_demo

,AAPL_first,MSFT_shifted,AAPL_plus_MSFT,AAPL_plus_MSFT_fill0
date,,,,
2003-01-02,7.4000,NaN,NaN,7.4000
2003-01-03,7.4500,NaN,NaN,7.4500
2003-01-06,7.4500,21.5200,28.9700,28.9700
2003-01-07,7.4300,21.9300,29.3600,29.3600
2003-01-08,7.2800,21.3100,28.5900,28.5900
2003-01-09,NaN,21.9300,NaN,21.9300
2003-01-10,NaN,21.9700,NaN,21.9700


In [31]:
# DataFrame끼리의 산술 연산도 행/열 이름을 기준으로 정렬됨
left = stock[["AAPL", "MSFT"]].head(3)
right = stock[["MSFT", "XOM"]].head(4)

print("left")
display(left)
print("right")
display(right)
print("left + right")
display(left + right)

left


,AAPL,MSFT
date,,
2003-01-02,7.4000,21.1100
2003-01-03,7.4500,21.1400
2003-01-06,7.4500,21.5200


right


,MSFT,XOM
date,,
2003-01-02,21.1100,29.2200
2003-01-03,21.1400,29.2400
2003-01-06,21.5200,29.9600
2003-01-07,21.9300,28.9500


left + right


,AAPL,MSFT,XOM
date,,,
2003-01-02,NaN,42.2200,NaN
2003-01-03,NaN,42.2800,NaN
2003-01-06,NaN,43.0400,NaN
2003-01-07,NaN,NaN,NaN


## 10. `apply`로 함수 적용하기

`apply`는 DataFrame의 열 또는 행 단위로 함수를 적용합니다.

- `axis=0`: 열 단위 적용
- `axis=1`: 행 단위 적용

In [32]:
cols = ["Price", "RM", "LSTAT", "AGE", "TAX"]
func = lambda x: x.max() - x.min()

# 열 단위로 범위(max-min) 계산
boston[cols].apply(func, axis=0)

Price    45.0000
RM        5.2190
LSTAT    36.2400
AGE      97.1000
TAX     524.0000
dtype: float64

In [33]:
# 행 단위로 사용자 정의 계산 적용
# 예: 가격 대비 방 수와 하위계층 비율을 함께 고려한 단순 지표
boston_eval = boston[cols].head(10).copy()
boston_eval["simple_score"] = boston_eval.apply(
    lambda row: row["Price"] / row["RM"] - row["LSTAT"] / 10,
    axis=1,
)
boston_eval

,Price,RM,LSTAT,AGE,TAX,simple_score
0,24.0000,6.5750,4.9800,65.2000,296.0000,3.1522
1,21.6000,6.4210,9.1400,78.9000,242.0000,2.4500
2,34.7000,7.1850,4.0300,61.1000,242.0000,4.4265
3,33.4000,6.9980,2.9400,45.8000,222.0000,4.4788
4,36.2000,7.1470,5.3300,54.2000,222.0000,4.5321
5,28.7000,6.4300,5.2100,58.7000,222.0000,3.9425
6,22.9000,6.0120,12.4300,66.6000,311.0000,2.5660
7,27.1000,6.1720,19.1500,96.1000,311.0000,2.4758
8,16.5000,5.6310,29.9300,100.0000,311.0000,-0.0628
9,18.9000,6.0040,17.1000,85.9000,311.0000,1.4379


In [34]:
# apply가 여러 값을 반환하는 Series를 만들 수도 있음
def summary_stats(s):
    return pd.Series({
        "min": s.min(),
        "mean": s.mean(),
        "max": s.max(),
        "range": s.max() - s.min(),
    })

boston[cols].apply(summary_stats, axis=0)

,Price,RM,LSTAT,AGE,TAX
min,5.0000,3.5610,1.7300,2.9000,187.0000
mean,22.5328,6.2846,12.6531,68.5749,408.2372
max,50.0000,8.7800,37.9700,100.0000,711.0000
range,45.0000,5.2190,36.2400,97.1000,524.0000


## 11. 정렬과 순위

- `sort_index()`: 인덱스 기준 정렬
- `sort_values()`: 값 기준 정렬
- `rank()`: 순위 계산

In [35]:
# 날짜 인덱스 기준 내림차순 정렬
stock.sort_index(ascending=False).head()

,AAPL,MSFT,XOM,SPX
date,,,,
2011-10-14,422.0000,27.2700,78.1100,1224.5800
2011-10-13,408.4300,27.1800,76.3700,1203.6600
2011-10-12,402.1900,26.9600,77.1600,1207.2500
2011-10-11,400.2900,27.0000,76.2700,1195.5400
2011-10-10,388.8100,26.9400,76.2800,1194.8900


In [36]:
# 값 기준 정렬: Boston 가격이 높은 상위 10개 행
boston.sort_values("Price", ascending=False).head(10)

,Price,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
283,50.0000,0.0150,90.0000,1.2100,1.0000,0.4010,7.9230,24.8000,5.8850,1.0000,198.0000,13.6000,395.5200,3.1600
225,50.0000,0.5269,0.0000,6.2000,0.0000,0.5040,8.7250,83.0000,2.8944,8.0000,307.0000,17.4000,382.0000,4.6300
369,50.0000,5.6700,0.0000,18.1000,1.0000,0.6310,6.6830,96.8000,1.3567,24.0000,666.0000,20.2000,375.3300,3.7300
370,50.0000,6.5388,0.0000,18.1000,1.0000,0.6310,7.0160,97.5000,1.2024,24.0000,666.0000,20.2000,392.0500,2.9600
371,50.0000,9.2323,0.0000,18.1000,0.0000,0.6310,6.2160,100.0000,1.1691,24.0000,666.0000,20.2000,366.1500,9.5300
372,50.0000,8.2673,0.0000,18.1000,1.0000,0.6680,5.8750,89.6000,1.1296,24.0000,666.0000,20.2000,347.8800,8.8800
186,50.0000,0.0560,0.0000,2.4600,0.0000,0.4880,7.8310,53.6000,3.1992,3.0000,193.0000,17.8000,392.6300,4.4500
204,50.0000,0.0201,95.0000,2.6800,0.0000,0.4161,8.0340,31.9000,5.1180,4.0000,224.0000,14.7000,390.5500,2.8800
257,50.0000,0.6115,20.0000,3.9700,0.0000,0.6470,8.7040,86.9000,1.8010,5.0000,264.0000,13.0000,389.7000,5.1200
195,50.0000,0.0138,80.0000,0.4600,0.0000,0.4220,7.8750,32.0000,5.6484,4.0000,255.0000,14.4000,394.2300,2.9700


In [37]:
# 여러 컬럼 기준 정렬: Iris 품종별, 꽃잎 길이 기준
iris.sort_values(["species", "petal_length"], ascending=[True, False]).head(10)

,sepal_length,sepal_width,petal_length,petal_width,species
24,4.8000,3.4000,1.9000,0.2000,Iris-setosa
44,5.1000,3.8000,1.9000,0.4000,Iris-setosa
5,5.4000,3.9000,1.7000,0.4000,Iris-setosa
18,5.7000,3.8000,1.7000,0.3000,Iris-setosa
20,5.4000,3.4000,1.7000,0.2000,Iris-setosa
23,5.1000,3.3000,1.7000,0.5000,Iris-setosa
11,4.8000,3.4000,1.6000,0.2000,Iris-setosa
25,5.0000,3.0000,1.6000,0.2000,Iris-setosa
26,5.0000,3.4000,1.6000,0.4000,Iris-setosa
29,4.7000,3.2000,1.6000,0.2000,Iris-setosa


In [38]:
# 가격 순위 계산
price_rank = boston[["Price", "RM", "LSTAT"]].copy()
price_rank["price_rank_desc"] = price_rank["Price"].rank(ascending=False, method="dense")
price_rank.sort_values("price_rank_desc").head(10)

,Price,RM,LSTAT,price_rank_desc
204,50.0000,8.0340,2.8800,1.0000
267,50.0000,8.2970,7.4400,1.0000
257,50.0000,8.7040,5.1200,1.0000
225,50.0000,8.7250,4.6300,1.0000
195,50.0000,7.8750,2.9700,1.0000
368,50.0000,4.9700,3.2600,1.0000
369,50.0000,6.6830,3.7300,1.0000
370,50.0000,7.0160,2.9600,1.0000
371,50.0000,6.2160,9.5300,1.0000
372,50.0000,5.8750,8.8800,1.0000


## 12. 기술통계와 요약 통계

DataFrame/Series는 기본적인 요약 통계 메서드를 제공합니다.

- `count`, `describe`, `quantile`
- `sum`, `mean`, `median`
- `var`, `std`
- `diff`: 1차 차분, 시계열 데이터에서 자주 사용

In [39]:
# Boston 주요 변수 요약 통계
boston[["Price", "RM", "LSTAT", "AGE", "TAX"]].describe()

,Price,RM,LSTAT,AGE,TAX
count,506.0000,506.0000,506.0000,506.0000,506.0000
mean,22.5328,6.2846,12.6531,68.5749,408.2372
std,9.1971,0.7026,7.1411,28.1489,168.5371
min,5.0000,3.5610,1.7300,2.9000,187.0000
25%,17.0250,5.8855,6.9500,45.0250,279.0000
50%,21.2000,6.2085,11.3600,77.5000,330.0000
75%,25.0000,6.6235,16.9550,94.0750,666.0000
max,50.0000,8.7800,37.9700,100.0000,711.0000


In [40]:
# 분위수 계산
boston[["Price", "RM", "LSTAT"]].quantile([0.25, 0.5, 0.75])

,Price,RM,LSTAT
0.2500,17.0250,5.8855,6.9500
0.5000,21.2000,6.2085,11.3600
0.7500,25.0000,6.6235,16.9550


In [41]:
# 각 변수의 평균, 중앙값, 분산, 표준편차
summary_table = pd.DataFrame({
    "mean": boston[["Price", "RM", "LSTAT"]].mean(),
    "median": boston[["Price", "RM", "LSTAT"]].median(),
    "var": boston[["Price", "RM", "LSTAT"]].var(),
    "std": boston[["Price", "RM", "LSTAT"]].std(),
})
summary_table

,mean,median,var,std
Price,22.5328,21.2000,84.5867,9.1971
RM,6.2846,6.2085,0.4937,0.7026
LSTAT,12.6531,11.3600,50.9948,7.1411


In [42]:
# Shampoo 판매량의 월별 차분
shampoo_sales = shampoo["sales"].copy()
shampoo_diff = pd.DataFrame({
    "sales": shampoo_sales,
    "diff": shampoo_sales.diff(),
    "pct_change": shampoo_sales.pct_change(),
})
shampoo_diff.head(12)

,sales,diff,pct_change
date,,,
2001-01-01,266.0000,NaN,NaN
2001-02-01,145.9000,-120.1000,-0.4515
2001-03-01,183.1000,37.2000,0.2550
2001-04-01,119.3000,-63.8000,-0.3484
2001-05-01,180.3000,61.0000,0.5113
2001-06-01,168.5000,-11.8000,-0.0654
2001-07-01,231.8000,63.3000,0.3757
2001-08-01,224.5000,-7.3000,-0.0315
2001-09-01,192.8000,-31.7000,-0.1412


In [43]:
# 주가 수익률 요약
stock_returns = stock.pct_change().dropna()
stock_returns.describe()

,AAPL,MSFT,XOM,SPX
count,2213.0000,2213.0000,2213.0000,2213.0000
mean,0.0021,0.0003,0.0006,0.0002
std,0.0245,0.0177,0.0167,0.0135
min,-0.1792,-0.1175,-0.1395,-0.0903
25%,-0.0110,-0.0079,-0.0076,-0.0053
50%,0.0019,0.0000,0.0010,0.0008
75%,0.0152,0.0081,0.0091,0.0059
max,0.1390,0.1859,0.1720,0.1158


In [44]:
# RAM 가격은 범위가 매우 넓으므로 로그 변환 후 요약
ram_work = ram_price.copy()
ram_work["log10_price"] = np.log10(ram_work["price"])
ram_work[["date", "price", "log10_price"]].head()

,date,price,log10_price
0,1957.0000,411041792.0000,8.6139
1,1959.0000,67947725.0000,7.8322
2,1960.0000,5242880.0000,6.7196
3,1965.0000,2642412.0000,6.4220
4,1970.0000,734003.0000,5.8657


## 13. 유일값 처리: `unique`, `value_counts`, `isin`

범주형 데이터의 분포를 확인할 때 자주 사용합니다.

In [45]:
# Iris 품종의 유일값과 빈도
print("unique:", iris["species"].unique())
display(iris["species"].value_counts())

unique: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [46]:
# Cancer 데이터의 type 분포
cancer_type_counts = pd.DataFrame({
    "count": cancer["type"].value_counts(),
    "ratio": cancer["type"].value_counts(normalize=True),
})
cancer_type_counts

,count,ratio
type,,
benign,357,0.6274
malignant,212,0.3726


In [47]:
# WDBC 진단값 M/B를 사람이 읽기 쉬운 라벨로 변환
wdbc_work = wdbc.copy()
wdbc_work["diagnosis_label"] = wdbc_work["diagnosis"].map({"M": "malignant", "B": "benign"})
wdbc_work[["id", "diagnosis", "diagnosis_label"]].head()

,id,diagnosis,diagnosis_label
0,842302,M,malignant
1,842517,M,malignant
2,84300903,M,malignant
3,84348301,M,malignant
4,84358402,M,malignant


In [48]:
# isin으로 특정 품종만 필터링
selected_iris = iris[iris["species"].isin(["Iris-virginica", "Iris-versicolor"])]
selected_iris["species"].value_counts()

species
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

## 14. 상관관계와 공분산

- `corr()`: 상관계수. 보통 -1에서 1 사이의 값으로 해석합니다.
- `cov()`: 공분산. 두 변수가 함께 변하는 방향과 크기를 나타내지만 값의 범위가 정해져 있지 않습니다.
- `corrwith()`: 특정 Series 또는 DataFrame과의 상관관계를 계산합니다.

In [49]:
# Boston 주요 변수의 상관관계
boston_corr = boston[["Price", "RM", "LSTAT", "AGE", "TAX"]].corr()
boston_corr

,Price,RM,LSTAT,AGE,TAX
Price,1.0000,0.6954,-0.7377,-0.3770,-0.4685
RM,0.6954,1.0000,-0.6138,-0.2403,-0.2920
LSTAT,-0.7377,-0.6138,1.0000,0.6023,0.5440
AGE,-0.3770,-0.2403,0.6023,1.0000,0.5065
TAX,-0.4685,-0.2920,0.5440,0.5065,1.0000


In [50]:
# Price와 다른 변수의 상관관계
price_corr = boston.select_dtypes("number").corr()["Price"].sort_values(ascending=False)
price_corr

Price      1.0000
RM         0.6954
ZN         0.3604
B          0.3335
DIS        0.2499
CHAS       0.1753
AGE       -0.3770
RAD       -0.3816
CRIM      -0.3858
NOX       -0.4273
TAX       -0.4685
INDUS     -0.4837
PTRATIO   -0.5078
LSTAT     -0.7377
Name: Price, dtype: float64

In [51]:
# Boston 주요 변수의 공분산
boston_cov = boston[["Price", "RM", "LSTAT", "AGE", "TAX"]].cov()
boston_cov

,Price,RM,LSTAT,AGE,TAX
Price,84.5867,4.4934,-48.4475,-97.5890,-726.2557
RM,4.4934,0.4937,-3.0797,-4.7519,-34.5834
LSTAT,-48.4475,-3.0797,50.9948,121.0777,654.7145
AGE,-97.5890,-4.7519,121.0777,792.3584,2402.6901
TAX,-726.2557,-34.5834,654.7145,2402.6901,28404.7595


In [52]:
# 주가 수익률 간 상관관계와 공분산
returns_corr = stock_returns[["AAPL", "MSFT", "XOM", "SPX"]].corr()
returns_cov = stock_returns[["AAPL", "MSFT", "XOM", "SPX"]].cov()

print("상관관계")
display(returns_corr)
print("공분산")
display(returns_cov)

상관관계


,AAPL,MSFT,XOM,SPX
AAPL,1.0000,0.4447,0.3859,0.5645
MSFT,0.4447,1.0000,0.5347,0.7148
XOM,0.3859,0.5347,1.0000,0.7646
SPX,0.5645,0.7148,0.7646,1.0000


공분산


,AAPL,MSFT,XOM,SPX
AAPL,0.0006,0.0002,0.0002,0.0002
MSFT,0.0002,0.0003,0.0002,0.0002
XOM,0.0002,0.0002,0.0003,0.0002
SPX,0.0002,0.0002,0.0002,0.0002


In [53]:
# Cancer 특성과 악성 여부의 상관관계
# type이 malignant이면 1, benign이면 0으로 변환한 뒤 수치 변수와 corrwith 수행
malignant_flag = (cancer["type"] == "malignant").astype(int)
cancer_corr_with_target = cancer.select_dtypes("number").corrwith(malignant_flag).sort_values(ascending=False)
cancer_corr_with_target.head(10)

worst concave points   0.7936
worst perimeter        0.7829
mean concave points    0.7766
worst radius           0.7765
mean perimeter         0.7426
worst area             0.7338
mean radius            0.7300
mean area              0.7090
mean concavity         0.6964
worst concavity        0.6596
dtype: float64

## 15. 종합 미션

아래 문제는 앞에서 다룬 기능을 조합하여 해결합니다.  
각 문제의 해설 코드는 바로 아래 셀에 포함되어 있으므로, 먼저 직접 작성해 본 뒤 비교해보세요.

### 미션 A. Boston 데이터
1. `Price`, `RM`, `LSTAT`, `TAX`만 선택하세요.
2. `Price`가 중앙값보다 큰 행만 필터링하세요.
3. `price_per_room = Price / RM` 컬럼을 추가하세요.
4. `price_per_room` 기준 상위 5개 행을 확인하세요.

In [54]:
# 미션 A 해설 코드
mission_a = boston[["Price", "RM", "LSTAT", "TAX"]].copy()
mission_a = mission_a[mission_a["Price"] > mission_a["Price"].median()]
mission_a["price_per_room"] = mission_a["Price"] / mission_a["RM"]
mission_a.sort_values("price_per_room", ascending=False).head(5)

,Price,RM,LSTAT,TAX,price_per_room
368,50.0000,4.9700,3.2600,666.0000,10.0604
372,50.0000,5.8750,8.8800,666.0000,8.5106
371,50.0000,6.2160,9.5300,666.0000,8.0438
365,27.5000,3.5610,7.1200,666.0000,7.7225
369,50.0000,6.6830,3.7300,666.0000,7.4817


### 미션 B. Iris 데이터
1. `species`별 데이터 수를 구하세요.
2. `sepal_length`가 전체 평균보다 큰 행만 선택하세요.
3. 선택된 결과에서 `species`, `sepal_length`, `petal_length`만 출력하세요.

In [55]:
# 미션 B 해설 코드
print("품종별 개수")
display(iris["species"].value_counts())

mission_b = iris[iris["sepal_length"] > iris["sepal_length"].mean()]
mission_b[["species", "sepal_length", "petal_length"]].head(10)

품종별 개수


species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

,species,sepal_length,petal_length
50,Iris-versicolor,7.0000,4.7000
51,Iris-versicolor,6.4000,4.5000
52,Iris-versicolor,6.9000,4.9000
54,Iris-versicolor,6.5000,4.6000
56,Iris-versicolor,6.3000,4.7000
58,Iris-versicolor,6.6000,4.6000
61,Iris-versicolor,5.9000,4.2000
62,Iris-versicolor,6.0000,4.0000
63,Iris-versicolor,6.1000,4.7000
65,Iris-versicolor,6.7000,4.4000


### 미션 C. Stock 데이터
1. 주가를 일별 수익률로 변환하세요.
2. AAPL 수익률의 평균, 표준편차, 최댓값, 최솟값을 계산하세요.
3. AAPL과 MSFT 수익률의 상관계수를 계산하세요.

In [56]:
# 미션 C 해설 코드
mission_c_returns = stock.pct_change().dropna()

summary_c = mission_c_returns["AAPL"].agg(["mean", "std", "min", "max"])
correlation_c = mission_c_returns["AAPL"].corr(mission_c_returns["MSFT"])

print("AAPL 수익률 요약")
display(summary_c)
print("AAPL-MSFT 수익률 상관계수:", correlation_c)

AAPL 수익률 요약


mean    0.0021
std     0.0245
min    -0.1792
max     0.1390
Name: AAPL, dtype: float64

AAPL-MSFT 수익률 상관계수: 0.44469678861732237


### 미션 D. Cancer 데이터
1. `type`의 빈도를 계산하세요.
2. `mean radius`, `mean texture`, `mean area`의 요약 통계를 확인하세요.
3. `type`을 `malignant=1`, `benign=0`으로 변환한 Series를 만들고 이름을 지정하세요.

In [57]:
# 미션 D 해설 코드
print("type 빈도")
display(cancer["type"].value_counts())

print("주요 변수 요약")
display(cancer[["mean radius", "mean texture", "mean area"]].describe())

type_flag = (cancer["type"] == "malignant").astype(int)
type_flag.name = "malignant_flag"
type_flag.index.name = "sample_id"
display(type_flag.head())

type 빈도


type
benign       357
malignant    212
Name: count, dtype: int64

주요 변수 요약


,mean radius,mean texture,mean area
count,569.0000,569.0000,569.0000
mean,14.1273,19.2896,654.8891
std,3.5240,4.3010,351.9141
min,6.9810,9.7100,143.5000
25%,11.7000,16.1700,420.3000
50%,13.3700,18.8400,551.1000
75%,15.7800,21.8000,782.7000
max,28.1100,39.2800,2501.0000


sample_id
0    1
1    1
2    1
3    1
4    1
Name: malignant_flag, dtype: int64

## 16. 점검용 체크리스트

아래 셀은 지금까지 만든 주요 객체가 의도대로 만들어졌는지 간단히 점검합니다.

In [58]:
assert isinstance(price_series, pd.Series)
assert isinstance(boston, pd.DataFrame)
assert boston.shape == (506, 14)
assert iris.shape[1] == 5
assert stock.index.name == "date"
assert "price_per_room" in boston_work.columns
assert cancer_filled[["mean radius", "mean texture", "mean area"]].isnull().sum().sum() == 0

print("점검 완료: 주요 실습 객체가 정상적으로 생성되었습니다.")

점검 완료: 주요 실습 객체가 정상적으로 생성되었습니다.


## 17. 마무리

이번 실습에서는 강의의 Pandas 시작하기, Series, DataFrame 파트를 실제 데이터로 재구성했습니다.

핵심 정리:

- Series는 1차원 데이터와 인덱스를 함께 가진 자료구조입니다.
- DataFrame은 행 인덱스와 열 인덱스를 가진 2차원 표형 자료구조입니다.
- pandas는 파일에서 데이터를 읽고, 선택하고, 결측을 처리하고, 통계 요약과 상관관계 분석까지 이어지는 실무 데이터 분석 흐름을 지원합니다.
- `loc`와 `iloc`, Boolean indexing, `apply`, `sort_values`, `describe`, `corr`, `cov`는 초급 실습에서 반드시 익숙해져야 할 기본 도구입니다.